# Python for Machine Learning — Practical Exercise


*Data Structures → Objects → Your First ML Pipeline*

---

**Learning goals for this notebook:**
- Use Python **dictionaries** to represent structured data
- Build **classes** to model real-world objects
- Understand the ML workflow: load → explore → preprocess → train → evaluate
- Apply these ideas first to *synthetic* data, then to a *real* dataset from Kaggle

> 💡 **Scaffolding tip:** Each section builds on the last. Complete the `# TODO` cells before moving forward. Solutions are hidden in collapsed cells at the end of each section.

---

## Part 0 - Setup env
Check uv at: https://iluvatar1.github.io/ML4Sci-lectures/lectures/01-reviewPython/PythonReview.html


## Part 0 — Setup imports
Run this cell first to import everything you'll need. This is mainly to check that the env is ok, but it is better to import stuff where you needed, sometimes

In [ ]:
# Standard library
import random
import math
from pprint import pprint

# Data & ML (install with: pip install numpy pandas scikit-learn matplotlib seaborn)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error

# Make outputs look nice
sns.set_theme(style='whitegrid')
random.seed(42)
np.random.seed(42)

print('✅ All imports successful!')


## Part 1 — Dictionaries: Storing One Record

A **dictionary** maps *keys* (labels) to *values* (data). Think of it like a row in a spreadsheet.

**Example — one student record:**

In [ ]:
# A single student described as a dictionary
student = {
    'name':        'Ana García',
    'age':         22,
    'study_hours': 6.5,   # hours per day
    'prev_grade':  78,    # previous exam score (0-100)
    'passed':      True   # did they pass the final exam?
}

# Access values by key
print('Name:', student['name'])
print(f"Study hours: {student['study_hours']}")

# .get() is safer — returns None if key doesn't exist
print(f"Scholarship: {student.get('scholarship', 'not recorded')}")

### Exercise — Create your own record
Create a dictionary called `my_student` representing a fictional student. Include at least 5 keys. Then print the student's name and whether they passed.

In [ ]:
# TODO: create my_student dictionary
my_student = {
    # your code here
}

# TODO: print name and passed status


<details>
<summary>💡 Click to reveal solution</summary>

```python
my_student = {
    'name':        'Carlos López',
    'age':         24,
    'study_hours': 4.0,
    'prev_grade':  65,
    'passed':      False
}
print(my_student['name'], '— passed:', my_student['passed'])
```
</details>


## Part 2 — Lists of Dictionaries: A Mini-Dataset

Real datasets have *many* rows. We can store them as a **list of dictionaries**.

In [ ]:
# ── Synthetic dataset: 10 students ───────────────────────────────────────────
def generate_students(n=10):
    """Generate n fake student records."""
    students = []
    for i in range(n):
        hours = round(random.uniform(1, 10), 1)
        prev  = random.randint(40, 95)
        # Simple rule: pass if study_hours > 4 AND prev_grade > 55
        passed = (hours > 4) and (prev > 55)
        students.append({
            'student_id':          i + 1,
            'study_hours': hours,
            'prev_grade':  prev,
            'passed':      int(passed)   # 1 = pass, 0 = fail
        })
    return students

dataset = generate_students(20)

# Show first 3 records
pprint(dataset[:3])

### Exercise — Explore the list
Using a loop (or list comprehension), answer:
1. How many students passed?
2. What is the average `study_hours` across all students?

In [ ]:
# TODO: count students who passed
passed_count = None  # replace with your code

# TODO: compute average study_hours
avg_hours = None     # replace with your code

print(f'Students who passed: {passed_count}')
print(f'Average study hours: {avg_hours:.2f}')

<details>
<summary>💡 Click to reveal solution</summary>

```python
passed_count = sum(s['passed'] for s in dataset)
avg_hours    = sum(s['study_hours'] for s in dataset) / len(dataset)
```
</details>


## Part 3 — Classes: Modeling a Student Object

A **class** bundles data (attributes) and behaviour (methods) together.
Think of it as a *blueprint* for creating many similar objects.

We'll create a `Student` class, then a `Dataset` class that holds many students.

In [ ]:
class Student:
    """Represents a single student with their features and label."""

    def __init__(self, student_id, study_hours, prev_grade, passed):
        self.student_id  = student_id
        self.study_hours = study_hours
        self.prev_grade  = prev_grade
        self.passed      = passed        # 1 or 0

    def is_at_risk(self):
        """Return True if the student studies < 3 hours or has prev_grade < 50."""
        return self.study_hours < 3 or self.prev_grade < 50

    def features(self):
        """Return a list of numeric features (used for ML models)."""
        return [self.study_hours, self.prev_grade]

    def label(self):
        """Return the target variable."""
        return self.passed

    def __repr__(self):
        status = 'PASS' if self.passed else 'FAIL'
        return (f'Student(id={self.student_id}, hours={self.study_hours}, '
                f'prev={self.prev_grade}, {status})')


# Build Student objects from our dictionary dataset
students = [Student(**{k: v for k, v in s.items()}) for s in dataset]

# Quick demo
s = students[0]
print(s)
print('At risk?', s.is_at_risk())
print('Features:', s.features())

### Exercise — Add a method to Student
Add a method called `risk_level()` to the `Student` class that returns:
- `'high'`   if study_hours < 3 **or** prev_grade < 50
- `'medium'` if study_hours < 5 **or** prev_grade < 65
- `'low'`    otherwise

Test it on a few students.

In [ ]:
class StudentV2(Student):   # inherits from Student
    """Extended Student with risk_level method."""

    def risk_level(self):
        # TODO: implement the logic described above
        pass


# Test
sv2 = StudentV2(student_id=99, study_hours=2.5, prev_grade=48, passed=0)
print(sv2.risk_level())   # expected: 'high'

<details>
<summary>💡 Click to reveal solution</summary>

```python
def risk_level(self):
    if self.study_hours < 3 or self.prev_grade < 50:
        return 'high'
    elif self.study_hours < 5 or self.prev_grade < 65:
        return 'medium'
    else:
        return 'low'
```
</details>


## Part 4 — Your First ML Pipeline (With synthetic data)

Machine learning in 4 steps:
1. **Prepare** — convert objects → arrays
2. **Split**   — train set / test set
3. **Train**   — fit a model on training data
4. **Evaluate**— measure accuracy on unseen test data

We'll use a **K-Nearest Neighbours (KNN)** classifier. The idea: predict a student's outcome by looking at the *k* most similar students.

In [ ]:
# ── Step 1: Prepare data ─────────────────────────────────────────────────────
# Generate a bigger synthetic dataset so we have enough to split
big_dataset = generate_students(200)

X = np.array([[s['study_hours'], s['prev_grade']] for s in big_dataset])
y = np.array([s['passed'] for s in big_dataset])

print('Feature matrix shape:', X.shape)   # (200, 2)
print('Label vector shape:  ', y.shape)   # (200,)
print('Class distribution   :', dict(zip(*np.unique(y, return_counts=True))))

In [ ]:
# ── Step 2: Train / Test split ────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {len(X_train)}')
print(f'Testing  samples: {len(X_test)}')

# Feature scaling — KNN is sensitive to scale
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
# ── Step 3: Train ─────────────────────────────────────────────────────────────
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

print('Model trained ✅')

In [ ]:
# ── Step 4: Evaluate ──────────────────────────────────────────────────────────
# https://en.wikipedia.org/wiki/Precision_and_recall?useskin=vector
# https://www.labelf.ai/blog/what-is-accuracy-precision-recall-and-f1-score
# ROC Curve? AUC?

y_pred = model.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.2%}\n')
print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

In [ ]:
# ── Visualise the decision boundary ──────────────────────────────────────────
# (plot on original-scale features for readability)
X_raw = np.array([[s['study_hours'], s['prev_grade']] for s in big_dataset])

h = 0.05
x_min, x_max = X_raw[:, 0].min() - 1, X_raw[:, 0].max() + 1
y_min, y_max = X_raw[:, 1].min() - 1, X_raw[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid_scaled = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
Z = model.predict(grid_scaled).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlGn')
scatter = ax.scatter(X_raw[:, 0], X_raw[:, 1], c=y, cmap='RdYlGn',
                     edgecolors='k', linewidths=0.5, s=60)
ax.set_xlabel('Study Hours / day')
ax.set_ylabel('Previous Grade')
ax.set_title('KNN Decision Boundary — Synthetic Student Data')
plt.colorbar(scatter, ax=ax, label='0=Fail  1=Pass')
plt.tight_layout()
plt.show()

### Exercise — Tune k
Loop over `k = [1, 3, 5, 7, 9, 11]`, train a KNN for each, and record test accuracy. Plot the results. Which value of k gives the best accuracy?

In [ ]:
k_values    = [1, 3, 5, 7, 9, 11]
accuracies  = []

for k in k_values:
    # TODO: train a KNeighborsClassifier with n_neighbors=k
    # TODO: evaluate on X_test, y_test
    # TODO: append accuracy to accuracies list
    pass

# TODO: plot k_values vs accuracies


<details>
<summary>💡 Click to reveal solution</summary>

```python
for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    accuracies.append(acc)

plt.figure(figsize=(7, 4))
plt.plot(k_values, accuracies, marker='o')
plt.xlabel('k (number of neighbours)')
plt.ylabel('Test Accuracy')
plt.title('KNN Accuracy vs k')
plt.show()

best_k = k_values[accuracies.index(max(accuracies))]
print(f'Best k = {best_k}  →  accuracy = {max(accuracies):.2%}')
```
</details>


## Part 5 — Real Data: Titanic Dataset (from Kaggle)

Now let's level up with a famous real-world dataset: the **Titanic** passenger survival dataset.

**Goal:** predict whether a passenger survived (`Survived = 1`) or not (`Survived = 0`) based on features like age, sex, and ticket class.

### Getting the Data

**Option A — Kaggle CLI** (recommended if you have a Kaggle account):
```bash
pip install kaggle
# Put your kaggle.json API key in ~/.kaggle/kaggle.json
kaggle competitions download -c titanic
unzip titanic.zip
```

**Option B — Direct URL** (no account needed — we use this below):

In [ ]:
# We load from a stable public mirror of the Titanic CSV
URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

df = pd.read_csv(URL)
print('Shape:', df.shape)
df.head()

### Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== dtypes & non-null counts ===')
df.info()
print('\n=== Missing values ===')
print(df.isnull().sum())
print('\n=== Summary statistics ===')
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Survival rate
df['Survived'].value_counts().plot.bar(ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_xticklabels(['Died', 'Survived'], rotation=0)
axes[0].set_title('Survival Count')

# Survival by sex
df.groupby('Sex')['Survived'].mean().plot.bar(ax=axes[1], color=['#3498db', '#e91e8c'])
axes[1].set_title('Survival Rate by Sex')
axes[1].set_ylabel('Survival Rate')
axes[1].set_xticklabels(['Female', 'Male'], rotation=0)

# Age distribution
df['Age'].dropna().plot.hist(ax=axes[2], bins=30, color='#9b59b6', edgecolor='white')
axes[2].set_title('Age Distribution')
axes[2].set_xlabel('Age')

plt.suptitle('Titanic — Exploratory Data Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Exercise — Class-level EDA
Create a bar chart showing the **survival rate by passenger class** (`Pclass`: 1, 2, or 3).

*Hint: use `df.groupby('Pclass')['Survived'].mean()`*

In [ ]:
# TODO: compute survival rate by Pclass
# TODO: plot as a bar chart



<details>
<summary>💡 Click to reveal solution</summary>

```python
survival_by_class = df.groupby('Pclass')['Survived'].mean()
survival_by_class.plot.bar(color=['gold', 'silver', '#cd7f32'], edgecolor='k')
plt.xticks([0,1,2], ['1st Class', '2nd Class', '3rd Class'], rotation=0)
plt.ylabel('Survival Rate')
plt.title('Survival Rate by Passenger Class')
plt.show()
```
</details>

### Preprocessing — wrapping logic in a class

Real data is messy. We'll build a `TitanicPreprocessor` class to handle it cleanly.

In [ ]:
class TitanicPreprocessor:
    """
    Encapsulates all data-cleaning steps for the Titanic dataset.
    This is a simplified version of a scikit-learn Pipeline.
    """

    FEATURES = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_encoded']
    TARGET   = 'Survived'

    def fit_transform(self, dataframe):
        """Clean the data and return X (features) and y (labels)."""
        df = dataframe.copy()

        # 1. Fill missing Age with median
        df['Age'] = df['Age'].fillna(df['Age'].median())

        # 2. Fill missing Fare with median
        df['Fare'] = df['Fare'].fillna(df['Fare'].median())

        # 3. Encode Sex: female → 1, male → 0
        df['Sex_encoded'] = (df['Sex'] == 'female').astype(int)

        # 4. Drop rows still missing target
        df = df.dropna(subset=[self.TARGET])

        X = df[self.FEATURES].values
        y = df[self.TARGET].values

        self._feature_names = self.FEATURES   # store for later inspection
        return X, y

    def feature_names(self):
        return self._feature_names


prep = TitanicPreprocessor()
X, y = prep.fit_transform(df)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Features used:', prep.feature_names())

### Train and Evaluate on Real Data

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Train
model = KNeighborsClassifier(n_neighbors=7)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2%}\n')
print(classification_report(y_test, y_pred, target_names=['Did not survive', 'Survived']))

### Exercise 6 — Feature engineering
Add a new feature: **`FamilySize = SibSp + Parch + 1`** (counting the passenger themselves).

1. Modify `TitanicPreprocessor` (or create `TitanicPreprocessorV2`) to include this feature.
2. Retrain and compare accuracy.

> **Think:** does a larger family help or hurt survival chances? Check the data first!

In [ ]:
class TitanicPreprocessorV2(TitanicPreprocessor):
    FEATURES = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_encoded', 'FamilySize']

    def fit_transform(self, dataframe):
        df = dataframe.copy()
        # TODO: add FamilySize column
        # TODO: call the parent's fit_transform logic OR reimplement it here
        pass

# TODO: retrain and print accuracy


<details>
<summary>💡 Click to reveal solution</summary>

```python
class TitanicPreprocessorV2(TitanicPreprocessor):
    FEATURES = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_encoded', 'FamilySize']

    def fit_transform(self, dataframe):
        df = dataframe.copy()
        df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
        df['Age']        = df['Age'].fillna(df['Age'].median())
        df['Fare']       = df['Fare'].fillna(df['Fare'].median())
        df['Sex_encoded']= (df['Sex'] == 'female').astype(int)
        df = df.dropna(subset=[self.TARGET])
        return df[self.FEATURES].values, df[self.TARGET].values

prep2 = TitanicPreprocessorV2()
X2, y2 = prep2.fit_transform(df)

X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42)
sc2 = StandardScaler()
X2_tr = sc2.fit_transform(X2_tr)
X2_te = sc2.transform(X2_te)

m2 = KNeighborsClassifier(n_neighbors=7)
m2.fit(X2_tr, y2_tr)
print(f'Accuracy with FamilySize: {accuracy_score(y2_te, m2.predict(X2_te)):.2%}')
```
</details>


## Bonus Challenge — Regression on House Prices

Classification predicts a *category*. **Regression** predicts a *number*.

**Dataset:** California Housing (built into scikit-learn — no download needed)

**Task:** Predict the **median house value** from features like income, rooms, etc.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df_house = housing.frame
print(df_house.shape)
df_house.head()

In [ ]:
# Quick correlation heatmap
plt.figure(figsize=(9, 6))
sns.heatmap(df_house.corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix — California Housing')
plt.tight_layout()
plt.show()

### Bonus Exercise
1. Split the housing data into train/test (80/20).
2. Train a `LinearRegression` model.
3. Compute the **Root Mean Squared Error (RMSE)** on the test set.
4. Plot **predicted vs actual** values on a scatter chart.

*Hint: `np.sqrt(mean_squared_error(y_true, y_pred))`*

In [ ]:
# Target column is 'MedHouseVal'
feature_cols = [c for c in df_house.columns if c != 'MedHouseVal']

X_h = df_house[feature_cols].values
y_h = df_house['MedHouseVal'].values

# TODO: split, scale, train LinearRegression, compute RMSE, scatter plot


<details>
<summary>💡 Click to reveal solution</summary>

```python
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

sc_h = StandardScaler()
Xh_tr = sc_h.fit_transform(Xh_tr)
Xh_te = sc_h.transform(Xh_te)

lr = LinearRegression()
lr.fit(Xh_tr, yh_tr)

yh_pred = lr.predict(Xh_te)
rmse = np.sqrt(mean_squared_error(yh_te, yh_pred))
print(f'RMSE: {rmse:.4f} (in $100,000 units)')

plt.figure(figsize=(6, 6))
plt.scatter(yh_te, yh_pred, alpha=0.3, s=10)
plt.plot([0, 5], [0, 5], 'r--', label='Perfect prediction')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Linear Regression: Predicted vs Actual')
plt.legend()
plt.show()
```
</details>


## Wrap-Up

**You have practised:**

| Concept | Where |
|---|---|
| Dictionaries | Parts 1 & 2 |
| Classes & Inheritance | Parts 3 & 5.3 |
| Data Exploration | Parts 2 & 5.2 |
| Train/Test Split | Parts 4 & 5.4 |
| Feature Scaling | Parts 4 & 5.4 |
| KNN Classification | Parts 4 & 5.4 |
| Feature Engineering | Exercise 6 |
| Linear Regression | Bonus |

**Next steps to explore:**
- Try a `DecisionTreeClassifier` or `RandomForestClassifier` from scikit-learn
- Look into **cross-validation** (`cross_val_score`) for more reliable evaluation
- Explore the [Kaggle Titanic competition](https://www.kaggle.com/c/titanic) to submit real predictions!

